# FiGS Examples Notebook
This notebook shows you how to convert a capture into a GSplat and then simulate a drone trajectory within it. The trajectory is flown by an MPC expert and can be viewed through a video output from the onboard camera of the drone.

Some useful settings for interactive work

In [13]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Importing the necessary libraries


In [14]:
import figs.render.capture_generation as pg
import figs.visualize.generate_videos as gv

from figs.simulator import Simulator
from figs.control.vehicle_rate_mpc import VehicleRateMPC

#### 1. Define the capture (video name in gsplats/capture)

In [15]:
capture_name = "backroom"    # Raw video; needs cell 2 (generate_gsplat) to build a GSplat from scratch
# capture_name = "button"      # Not included in the example download - no capture/output exists for this
# capture_name = "mid_gate"      # Pre-built scene from the example download; skip cell 2, it's already trained

#### 2. Generate FiGS environment from capture

In [16]:
# Only needed when capture_name is a raw video (e.g. "backroom") that hasn't been trained into a GSplat yet.
# mid_gate already has a trained GSplat in gsplats/workspace/outputs/mid_gate, so this is skipped.
pg.generate_gsplat(capture_name)

Output()

[11:24:33] 🎉 Done copying images with prefix 'frame_'.                                        ]8;id=4596048;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=4596049;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/process_data_utils.py#348\348]8;;\

[2026/09/28 11:24:34 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-r1024',
 'preprocessing': {'grayscale': True, 'resize_max': 1024}}
[2026/09/28 11:24:34 hloc INFO] Skipping the extraction.
[2026/09/28 11:24:34 hloc INFO] Found 44850 pairs.
[2026/09/28 11:24:34 hloc INFO] Matching local features with configuration:
{'model': {'name': 'superglue',
           'sinkhorn_iterations': 50,
           'weights': 'outdoor'},
 'output': 'matches-superglue'}
[2026/09/28 11:24:37 hloc INFO] Skipping the matching.
[2026/09/28 11:24:37 hloc WARNING] The database already exists, deleting it.
[2026/09/28 11:24:37 hloc INFO] Creating an empty database...
[2026/09/28 11:24:37 hloc INFO] Importing images into the database...
[2026/09/28 11:24:48 hloc INFO] Importing features into the database...
100%|██████████| 300/300 [00:00<00:00, 903.34it/s] 
[2026/09/28 11:24:48 hloc INFO] Import

Output()

[11:34:17] 🎉 🎉 🎉 All DONE 🎉 🎉 🎉                                                ]8;id=4596056;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/images_to_nerfstudio_dataset.py\images_to_nerfstudio_dataset.py]8;;\:]8;id=4596057;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/images_to_nerfstudio_dataset.py#135\135]8;;\

           Starting with 300 images                                                  ]8;id=4596063;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/images_to_nerfstudio_dataset.py\images_to_nerfstudio_dataset.py]8;;\:]8;id=4596064;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/images_to_nerfstudio_dataset.py#138\138]8;;\

           Colmap matched 300 images                                                 ]8;id=4596069;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/images_to_nerfstudio_dataset.py\images_to_nerfstudio_dataset.py]8;;\:]8;id=4596070;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/images_to_nerfstudio_dataset.py#138\138]8;;\

           COLMAP found poses for all images, CONGRATS!                              ]8;id=4596075;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/images_to_nerfstudio_dataset.py\images_to_nerfstudio_dataset.py]8;;\:]8;id=4596076;file:///home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/process_data/images_to_nerfstudio_dataset.py#138\138]8;;\

RANSAC: 16 inliers out of 20 points with threshold 0.05m
Command failed.
/home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/torch/utils/cpp_extension.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]
Traceback (most recent call last):
  File "/home/yutharsan/miniconda3/envs/kitchen/bin/ns-train", line 6, in <module>
    sys.exit(entrypoint())
  File "/home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/scripts/train.py", line 262, in entrypoint
    main(
  File "/home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/scripts/train.py", line 247, in main
    launch(
  File "/home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/nerfstudio/scri

#### 3. Define the scene and course

In [17]:
# scene_name, course_name = "button", "button_prod"       # A short test trajectory where the drone approaches a button
# scene_name, course_name = "mid_gate", "traverse"        # Robustness trajecytory in the SOUS VIDE paper
scene_name, course_name = "backroom", "circuit"         # Cluttered trajectory in the SOUS VIDE paper - needs cell 2 run first (raw capture, not pre-trained)

#### 4. Simulate within the FiGS environment

In [19]:
# Initialize the simulator and controller
sim = Simulator(scene_name,"eval_single","carl")
ctl = VehicleRateMPC("Viper",course_name,"carl")

# Use the ideal trajectory in VehicleRateMPC to get initial conditions and final time
t0,tf,x0 = ctl.tXUd[0,0],ctl.tXUd[-1,0],ctl.tXUd[0,1:11]

# Simulate the policy
Tro,Xro,Uro,Fro,Rgb,Dpt,Aux = sim.simulate(ctl,t0,tf,x0)

# Output the results as a video
gv.images_to_mp4(Rgb,course_name+'.mp4', ctl.hz)

Multiple configurations found under '/home/yutharsan/FYP/SousVide/gsplats/workspace/outputs/backroom'; using the most recent one: '/home/yutharsan/FYP/SousVide/gsplats/workspace/outputs/backroom/splatfacto/2026-09-28_113447/config.yml'.


Loading latest checkpoint from load_dir

✅ Done loading checkpoint from outputs/backroom/splatfacto/2026-09-28_113447/nerfstudio_models/step-000002000.ckpt

 If there is an incompatibility with the CasADi generated code, please consider changing your CasADi version.
Version 3.8.1 currently in use.


/home/yutharsan/miniconda3/envs/kitchen/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


 If there is an incompatibility with the CasADi generated code, please consider changing your CasADi version.
Version 3.8.1 currently in use.
